In [1]:
!pip install --upgrade pymupdf
from typing import List, Dict, Tuple, Optional
import fitz  # PyMuPDF

In [2]:
from dataclasses import dataclass

@dataclass
class Block:
    x0: int
    y0: int
    x1: int
    y1: int
    text: str
    block_number: int
    block_type: int
    
TEXT_BLOCK_TYPE: int = 0

In [3]:
from dataclasses import dataclass

@dataclass
class TextSegment:
    x0: int
    y0: int
    x1: int
    y1: int
    text: str
    font: str
    font_size: int
    block_id: int
    page_no: int

@dataclass
class TextBlock:
    x0: int
    y0: int
    x1: int
    y1: int
    text: str
    font: str
    font_size: int
    block_id: int
    page_no: int




In [4]:
from collections import Counter
from fitz import Page

# def is_intorduction_on_page(block: dict) -> Optional[List[Tuple[str, int]]]:
#     if block["type"] != TEXT_BLOCK_TYPE:
#         return None
    
#     line_with_intro: bool = False
    
#     for line in block["lines"]:
#         if line_with_intro:
            
        
#         for span in line["spans"]:
#            if "introduction" in span["text"].lower():
#                line_with_intro = True
               
#         if line_with_intro:
#             font_data: List[Tuple[str, int]] = []
#             for span in line["spans"]:
#                 font_data.append((span["font"], round(span["size"], 1)))

#             print("Introductoin found;")
#             print("INTRO DATA:")
#             print(font_data)
#             return font_data
            
#     return None
    
# def try_using_introduction(page: Page) -> Optional[Tuple[str, int]]:
#     for line in block["lines"]:
#         for span in line["spans"]:
#            if 

def get_font_and_size_from_block(block: Dict) -> Optional[Tuple[List[Tuple[str, int]], bool]]:
    if block["type"] != TEXT_BLOCK_TYPE:
        return None

    font_data: List[Tuple[str, int]] = []
    line_with_intro: bool            = False
    
    for line in block["lines"]:
        for span in line["spans"]:
            if "introduction" in span["text"].lower():
                line_with_intro = True
            else:
                font_data.append((span["font"], round(span["size"], 1)))
            # print(f"FROM FIUNCTION: {span["font"]}, {round(span["size"], 1)}")
            
    return (font_data, line_with_intro)

def get_most_likley_font_and_size(pdf_path: str) -> Tuple[str, int]:
    doc = fitz.open(pdf_path)

    font_size_counter: Counter = Counter()

    # ommit last and first page [ LATER ]
    for page in doc:        
        blocks = page.get_text("dict")["blocks"]

        for block in blocks:            
            data = get_font_and_size_from_block(block)
            if data:
                if data[1]: # if is intro
                    print("INTRO FOUND\nDATA:")
                    print(data[0])
                    return Counter(data[0]).most_common(1)[0][0]
                font_size_counter.update(data[0])

    most_common_font_and_size: Tuple[str, int] = font_size_counter.most_common(1)[0][0]
    
    return most_common_font_and_size
    
        

In [5]:
def get_text_block(block: Dict, page_number: int) -> Optional[List[TextSegment]]:
    if block["type"] != TEXT_BLOCK_TYPE:
        return None

    filtered_segments: List[TextSegment] = []
    for line in block["lines"]:
        for span in line["spans"]:            
            filtered_segments.append(
                TextSegment(
                    *span["bbox"],
                    span["text"],
                    span["font"],
                    round(span["size"], 1),
                    block["number"],
                    page_number
                )
            )
            
    return filtered_segments

def get_text_segments(pdf_path: str) -> List[TextSegment]:
    doc = fitz.open(pdf_path)

    segments: List[TextSegment] = []
    
    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        
        for block in blocks:
            s = get_text_block(block, page.number)
            if s:
                segments.extend(s)

    return segments            

In [6]:
def replace_special_characters(text: str) -> str:
    text = text.replace('\xad', '')
    text = text.replace('\u00ad', '')
    text = text.replace('\N{SOFT HYPHEN}', '')
    return text

In [7]:
from typing import List
from collections import defaultdict

def aggregate_segments_to_blocks(segments: List[TextSegment]) -> List[TextBlock]:
    """
    Groups individual TextSegments by page and block_id, 
    merging them into complete TextBlock objects.
    """
    # Group segments using a dictionary keyed by (page_no, block_id)
    grouped_segments = defaultdict(list)
    for seg in segments:
        grouped_segments[(seg.page_no, seg.block_id)].append(seg)
    
    blocks = []
    for (page_no, block_id), segs in grouped_segments.items():
        # Identify the bounding box for the entire block
        x0 = min(s.x0 for s in segs)
        y0 = min(s.y0 for s in segs)
        x1 = max(s.x1 for s in segs)
        y1 = max(s.y1 for s in segs)
        
        # Join text - assuming segments in the list are already in reading order 
        # from PyMuPDF's internal extraction.
        full_text = " ".join(s.text.strip() for s in segs if s.text.strip())
        
        # Take the font and size of the first segment as representative of the block
        representative = segs[0]
        
        blocks.append(
            TextBlock(
                x0=x0, 
                y0=y0, 
                x1=x1, 
                y1=y1,
                text=full_text,
                font=representative.font,
                font_size=representative.font_size,
                block_id=block_id,
                page_no=page_no
            )
        )
    return blocks
    
def sort_blocks_columnar_order(blocks: List[TextBlock], column_margin: int = 50) -> List[TextBlock]:
    """
    Sorts TextBlocks by page, then by column (Left-to-Right), 
    then top-to-bottom within each column.
    
    Args:
        blocks: List of aggregated TextBlock objects.
        column_margin: Pixel threshold to group blocks into the same column. 
                       50-100 is usually ideal for standard PDFs to ignore minor indentations.
    """
    return sorted(blocks, key=lambda b: (
        b.page_no, 
        b.x0 // column_margin,  # Secondary Sort: Groups x0 into distinct vertical "columns"
        b.y0                    # Tertiary Sort: Top-to-bottom inside that specific column
    ))

def get_processed_blocks(pdf_path: str) -> List[TextBlock]:
    """
    Main pipeline to execute the extraction, aggregation, and sorting.
    """
    most_likley_font_and_size: Tuple[str, int] = get_most_likley_font_and_size(pdf_file)
    segments = get_text_segments(pdf_path)

    filtered_segments = list(filter(
        lambda s: s.font == most_likley_font_and_size[0] and s.font_size == most_likley_font_and_size[1],     
        segments
    ))
    
    blocks = aggregate_segments_to_blocks(filtered_segments)
    return sort_blocks_columnar_order(blocks, column_margin=50)

def get_glued_str_only_blocks(sorted_blocks: List[TextBlock]) -> List[str]:
    output_string_blocks: List[str] = []
    
    block_index: int = 0
    current_tmp: str = ''
    while block_index < len(sorted_blocks):
        current_tmp += sorted_blocks[block_index].text + ' '
        if sorted_blocks[block_index].text.endswith("."):
            output_string_blocks.append(replace_special_characters(current_tmp.strip()))
            current_tmp = ''

        block_index += 1

    if current_tmp:
        output_string_blocks.append(replace_special_characters(current_tmp.strip()))
    
    return output_string_blocks
            
        # if not sorted_blocks[block_index].text.endswith("."):
        #     current_tmp += sorted_blocks[block_index].text
        # else
        #     output_string_blocks
            
    

In [8]:

def scrape_pdf(pdf_path: str) -> List[str]:
    doc = fitz.open(pdf_path)
    # for page in doc:
    page = doc[0]
    # OK
    # print(
    #     "\n".join(page.get_text().splitlines())
    # )

    blocks: List[Block] = list(map(lambda b: Block(*b),  page.get_text("blocks")))

    blocks_text_only: List[Block] = list(filter(lambda b: b.block_type == TEXT_BLOCK_TYPE, blocks))

    blocks_sorted: List[Block] = sorted(
        blocks_text_only,
        key = lambda block: block.y0
    )
    
    blocks_sorted = sorted(
        blocks_text_only,
        key = lambda block: block.x0
    )
    
    print('\n'.join(map(str, blocks_sorted)))
    
    return []
    

In [9]:
# pdf_file: str = "10.1148@rg.2019180069.pdf"
# pdf_file: str = "feister-et-al-2023-cardiothoracic-conditions-with-female-preponderance-case-based-imaging-review.pdf"
pdf_file: str = "el-sherief2014.pdf"

# print(scrape_pdf(pdf_file))

# most_likley_font_and_size: Tuple[str, int] = get_most_likley_font_and_size(pdf_file)
# print(most_likley_font_and_size)

# segments = get_text_segments(pdf_file)
# filtered_segments = list(filter(
#     lambda s: s.font == most_likley_font_and_size[0] and s.font_size == most_likley_font_and_size[1],     
#     segments
# ))

# print(
#     "\n\n".join(map(
#         lambda s: f"PAGE NO. [{s.page_no}][{s.block_id}] {s.text}",
#         filtered_segments
#     ))
# )

# print(
#     "\n\n".join(map(
#         lambda s: f"PAGE NO. [{s.page_no}][{s.block_id}] {s.text}",
#         get_processed_blocks(pdf_file)
#     ))
# )

with open("output.txt", "w+", encoding="utf-8") as output_file:
    output_file.write(
        "\n\n".join(
            get_glued_str_only_blocks(get_processed_blocks(pdf_file))
        )
    )



print("Done! Text saved to output.txt")

INTRO FOUND
DATA:
[('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0), ('Plantin', 10.0)]
Done! Text saved to output.txt
